# Lab 2: Meta-features

**Student:** Lopatenko George (334771)  
**Course:** Automated Machine Learning

This study asks whether inexpensive descriptions of a classification dataset can predict
which member of a fixed algorithm portfolio will achieve the highest balanced accuracy.
I compare hand-designed meta-features with PyMFE on 300 OpenML tasks, check that both
representations are invariant to arbitrary data ordering, and evaluate the resulting
meta-learning problems against a naive majority-class rule.

## 0. Reproducible experiment state

Dataset downloads, meta-feature extraction, and the base-level cross-validation are the
expensive part of this work. They are executed by `run_meta_experiments.py`, one OpenML
dataset per MLflow run. A completed dataset ID is never trained again; this notebook queries
the versioned MLflow records and rebuilds only the inexpensive analysis.

```bash
cd automl/lab-2
.venv/bin/python -u run_meta_experiments.py --target 300
.venv/bin/mlflow ui --backend-store-uri sqlite:///mlflow.db
```

The exported CSV is a readable snapshot, not a second source of truth. Re-running its cell
refreshes it from MLflow; if the local database is unavailable in a submitted copy, the
notebook uses that snapshot and never starts training implicitly.

In [1]:
import logging
import shutil
import sys
from pathlib import Path

import kaleido
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
from prettytable import PrettyTable

LAB_DIR = Path.cwd()
if (LAB_DIR / 'automl/lab-2').is_dir():
    LAB_DIR = LAB_DIR / 'automl/lab-2'
sys.path.insert(0, str(LAB_DIR))

from lab2_analysis import (
    PYMFE_NUMERICAL_ARTIFACTS,
    characterize_meta_datasets,
    evaluate_both_sources,
    extreme_feature_rows,
    feature_columns,
    pca_projection,
)
from lab2_experiment import (
    ALGORITHMS,
    MANUAL_FEATURE_DESCRIPTIONS,
    PYMFE_FEATURES,
    TARGET_DATASETS,
    export_completed_runs,
    load_invariance_summary,
    verify_score_selection,
)

RESULTS_DIR, FIGURES_DIR = LAB_DIR / 'results', LAB_DIR / 'figures'
for folder in [RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

COLORS = {
    'Logistic regression': '#0072B2',
    'RBF SVM': '#D55E00',
    'Random forest': '#009E73',
}
SYMBOLS = {'Logistic regression': 'circle', 'RBF SVM': 'diamond', 'Random forest': 'square'}
SOURCE_COLORS = {'Manual': '#5B4B8A', 'PyMFE': '#D17A22'}
pio.templates.default = 'plotly_white'
logging.getLogger('mlflow').setLevel(logging.ERROR)
logging.getLogger('alembic').setLevel(logging.ERROR)
logging.getLogger('alembic.runtime.migration').setLevel(logging.ERROR)


def show_table(frame, caption, precision=4, max_width=34):
    def format_value(value):
        if pd.isna(value):
            return '—'
        if isinstance(value, (float, np.floating)):
            return f'{value:.{precision}f}'
        return str(value)

    table = PrettyTable()
    table.title = caption
    table.field_names = [str(column) for column in frame.columns]
    table.align = 'l'
    for field in table.field_names:
        table.max_width[field] = max_width
    for row in frame.itertuples(index=False, name=None):
        table.add_row([format_value(value) for value in row])
    print(table)


def show_and_save(fig, stem, height=760):
    fig.update_layout(
        template='plotly_white', width=1400, height=height,
        font=dict(family='Arial', size=17, color='#243447'),
        margin=dict(l=90, r=55, t=125, b=185),
        title_y=0.95, title_yanchor='top',
        legend=dict(
            orientation='h', x=0, y=-0.12, xanchor='left', yanchor='top',
            title_text=''
        ),
    )
    fig.add_annotation(
        text='Source: OpenML tasks; experiments logged by this notebook project',
        x=0, y=0, yshift=-118, xref='paper', yref='paper', showarrow=False,
        xanchor='left', yanchor='top', font=dict(size=12, color='#536170'),
    )
    fig.update_xaxes(automargin=True)
    fig.update_yaxes(automargin=True)
    path = FIGURES_DIR / f'{stem}.png'
    fig.write_image(str(path), width=1400, height=height, scale=3)
    fig.write_image(FIGURES_DIR / f'{stem}.svg', width=1400, height=height)
    fig.write_html(FIGURES_DIR / f'{stem}.html', include_plotlyjs=True)
    fig.write_json(FIGURES_DIR / f'{stem}.json')
    fig.show(config={'displaylogo': False, 'responsive': True})
    return path


logging.disable(logging.INFO)
logged_meta_dataset = export_completed_runs()
logging.disable(logging.NOTSET)
if len(logged_meta_dataset) >= TARGET_DATASETS:
    meta_dataset = logged_meta_dataset.head(TARGET_DATASETS).copy()
    data_source = 'MLflow'
else:
    snapshot = pd.read_csv(RESULTS_DIR / 'meta_dataset_from_mlflow.csv')
    meta_dataset = snapshot.head(TARGET_DATASETS).copy()
    data_source = 'the last MLflow export'
if len(meta_dataset) < TARGET_DATASETS:
    raise RuntimeError(
        f'Only {len(meta_dataset)} completed runs are available. '
        'Resume run_meta_experiments.py before executing the report.'
    )
if not verify_score_selection(meta_dataset):
    raise RuntimeError('A stored winner label disagrees with the logged base scores.')

manual_columns, pymfe_columns = feature_columns(meta_dataset)
print(f'Loaded {len(meta_dataset)} completed dataset runs from {data_source}.')

Matplotlib is building the font cache; this may take a moment.


Loaded 300 completed dataset runs from MLflow.


## 1. Dataset cohort and base-level protocol

I select active OpenML supervised-classification tasks with 100–5,000 rows, 2–100
predictors, 2–10 classes, at least six observations in the smallest class, no more than
30 categorical predictors, and no categorical predictor with more than 50 levels. One
task is retained per dataset name and the eligible catalog is shuffled with seed 334771.
The first 300 tasks that can be processed form the cohort.

Each dataset is evaluated on the same three stratified folds with **balanced accuracy**.
The portfolio contains class-weighted one-vs-rest logistic regression, a class-weighted
RBF SVM, and a class-weighted random forest. Numeric values are median-imputed and scaled;
categorical values are mode-imputed and one-hot encoded inside each fold. The meta-class
is the algorithm with the highest mean validation score. No dataset receives individual
hyperparameter tuning.

In [2]:
score_columns = {name: 'score__' + name.replace(' ', '_') for name in ALGORITHMS}
winner_counts = meta_dataset['best_algorithm'].value_counts().reindex(ALGORITHMS, fill_value=0)

cohort_summary = pd.DataFrame([
    {'Quantity': 'Datasets', 'Value': len(meta_dataset)},
    {'Quantity': 'Rows per dataset, median', 'Value': meta_dataset['n_rows'].median()},
    {'Quantity': 'Rows per dataset, range',
     'Value': f"{int(meta_dataset['n_rows'].min())}–{int(meta_dataset['n_rows'].max())}"},
    {'Quantity': 'Predictors per dataset, median', 'Value': meta_dataset['n_features'].median()},
    {'Quantity': 'Predictors per dataset, range',
     'Value': f"{int(meta_dataset['n_features'].min())}–{int(meta_dataset['n_features'].max())}"},
    {'Quantity': 'Total logged runtime, minutes', 'Value': meta_dataset['total_seconds'].sum() / 60},
])
show_table(cohort_summary, 'Final OpenML cohort and measured workload', precision=2)

+-------------------------------------------+
| Final OpenML cohort and measured workload |
+--------------------------------+----------+
| Quantity                       | Value    |
+--------------------------------+----------+
| Datasets                       | 300      |
| Rows per dataset, median       | 500.00   |
| Rows per dataset, range        | 100–5000 |
| Predictors per dataset, median | 10.00    |
| Predictors per dataset, range  | 2–90     |
| Total logged runtime, minutes  | 21.52    |
+--------------------------------+----------+


The final meta-dataset contains exactly **300 datasets**. Its rows are
datasets, not original observations: an OpenML task with thousands of cases contributes
one meta-example. The filters keep the base experiment feasible but deliberately retain
mixed numerical/categorical data, missing values, binary targets, and multiclass targets.
This diversity is important because a meta-model trained only on uniformly clean datasets
would learn little about algorithm preference.

## 2. Hand-designed meta-features

The hand-designed representation contains basic size/type counts, distributional summaries,
and structural descriptors. All vector-valued quantities are reduced symmetrically across
predictors, so no coordinate depends on which column happens to appear first. Numerical PCA
is performed after median imputation and standardization. The Fisher ratio compares
between-class with within-class variation; it is an association measure, not a claim that
class boundaries are linear.

In [3]:
feature_catalog = pd.DataFrame([
    {'Group': group, 'Meta-feature': name.replace('__', ' / '), 'Meaning': meaning}
    for name, (group, meaning) in MANUAL_FEATURE_DESCRIPTIONS.items()
])
show_table(feature_catalog, 'Hand-designed meta-features', max_width=48)

+------------------------------------------------------------------------------------------------------+
|                                     Hand-designed meta-features                                      |
+-------------+-------------------------------------+--------------------------------------------------+
| Group       | Meta-feature                        | Meaning                                          |
+-------------+-------------------------------------+--------------------------------------------------+
| Basic       | basic / n_instances                 | Number of usable rows                            |
| Basic       | basic / n_features                  | Number of non-constant predictors                |
| Basic       | basic / n_numeric                   | Number of numerical predictors                   |
| Basic       | basic / n_categorical               | Number of categorical predictors                 |
| Basic       | basic / n_classes                   | N

`missing_fraction` is measured before imputation. \
Correlation and shape summaries use numerical predictors only. \
Categorical structure is represented through the number and cardinality of categorical variables, while duplicated-row and distinct-value fractions use the full mixed-type table. 

## 3. Invariance to arbitrary representation choices

In [4]:
invariance = load_invariance_summary()
invariance['Passed'] = invariance['Invariant'].astype(str) + '/' + invariance['Features'].astype(str)
invariance['Max |Δ|'] = invariance['max_abs_difference'].map(lambda value: f'{value:.2e}')
show_table(
    invariance[['Extractor', 'Variant', 'Passed', 'Max |Δ|']],
    'Row, column, and category-order invariance on credit-g',
)

+--------------------------------------------------------+
| Row, column, and category-order invariance on credit-g |
+------------+-------------------+---------+-------------+
| Extractor  | Variant           | Passed  | Max |Δ|     |
+------------+-------------------+---------+-------------+
| Manual     | category labels   | 19/19   | 0.00e+00    |
| Manual     | column order      | 19/19   | 2.22e-16    |
| Manual     | row order         | 19/19   | 3.61e-16    |
| PyMFE      | category labels   | 26/26   | 4.44e-16    |
| PyMFE      | column order      | 26/26   | 3.55e-15    |
| PyMFE      | row order         | 26/26   | 3.55e-15    |
+------------+-------------------+---------+-------------+


Every comparison passes. The largest observed absolute discrepancy is
**3.55e-15**, which is floating-point roundoff rather than a material
change. This check also revealed why PyMFE is configured with `one-hot-full`: its shorter
`k-1` encoding chooses a reference category, so category relabeling can change downstream
statistical summaries. Retaining all indicator columns removes that arbitrary choice.

## 4. Base algorithms and meta-classes

In [5]:
winner_summary = pd.DataFrame([
    {
        'Algorithm': name,
        'Datasets won': int(winner_counts[name]),
        'Win share': winner_counts[name] / len(meta_dataset),
        'Median balanced accuracy': meta_dataset[column].median(),
        'Mean balanced accuracy': meta_dataset[column].mean(),
    }
    for name, column in score_columns.items()
])
show_table(winner_summary, 'Base-level performance and meta-class balance')

fig = go.Figure()
for name, column in score_columns.items():
    fig.add_trace(go.Box(
        y=meta_dataset[column], name=name, legendgroup=name, showlegend=True,
        marker_color=COLORS[name], line=dict(width=2), boxpoints='outliers',
        hovertemplate=f'{name}<br>Balanced accuracy=%{{y:.3f}}<extra></extra>',
    ))
fig.update_yaxes(title='Three-fold balanced accuracy', range=[0, 1.02], tickformat='.0%')
fig.update_xaxes(title='Fixed base algorithm')
fig.update_layout(
    title='Random forest has the highest median score, but distributions overlap<br>'
          '<sup>One score per OpenML task · identical folds within each task</sup>',
    legend_title_text='Base algorithm',
)
_ = show_and_save(fig, '01_base_algorithm_performance')

+----------------------------------------------------------------------------------------------------+
|                           Base-level performance and meta-class balance                            |
+---------------------+--------------+-----------+--------------------------+------------------------+
| Algorithm           | Datasets won | Win share | Median balanced accuracy | Mean balanced accuracy |
+---------------------+--------------+-----------+--------------------------+------------------------+
| Logistic regression | 95           | 0.3167    | 0.7027                   | 0.6997                 |
| RBF SVM             | 99           | 0.3300    | 0.7108                   | 0.7021                 |
| Random forest       | 106          | 0.3533    | 0.7575                   | 0.7277                 |
+---------------------+--------------+-----------+--------------------------+------------------------+


**Random forest** wins most often (106 of 300 datasets),
but the other algorithms still define non-empty meta-classes. Median balanced accuracies
are Logistic regression 0.703, RBF SVM 0.711, Random forest 0.757. On
**34.7%** of datasets, the two leading algorithms are within 0.01 balanced-
accuracy points. Those near-ties make the label intrinsically unstable: the meta-model is
predicting the winner under this finite CV protocol, not a universal property of a dataset.

## 5. Manual meta-dataset: two-dimensional projection

In [6]:
manual_projection, manual_variance, manual_loadings = pca_projection(meta_dataset, manual_columns)
manual_projection.to_csv(RESULTS_DIR / 'manual_projection.csv', index=False)
manual_loadings.to_csv(RESULTS_DIR / 'manual_pca_loadings.csv', index=False)

fig = go.Figure()
for algorithm in ALGORITHMS:
    subset = manual_projection[manual_projection['best_algorithm'].eq(algorithm)]
    fig.add_trace(go.Scatter(
        x=subset['PC1'], y=subset['PC2'], mode='markers', name=algorithm,
        legendgroup=algorithm,
        marker=dict(color=COLORS[algorithm], symbol=SYMBOLS[algorithm], size=10,
                    opacity=1.0, line=dict(color='white', width=0.7)),
        customdata=subset[['dataset_name', 'dataset_id']],
        hovertemplate='%{customdata[0]} · OpenML %{customdata[1]}<br>'
                      'PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra></extra>',
    ))
fig.update_xaxes(title=f'PC1 ({manual_variance[0]:.1%} variance)')
fig.update_yaxes(title=f'PC2 ({manual_variance[1]:.1%} variance)')
fig.update_layout(
    title='Hand-designed meta-features reveal overlapping algorithm-preference regions<br>'
          '<sup>Median imputation and standardization precede PCA</sup>',
    legend_title_text='Best base algorithm',
)
_ = show_and_save(fig, '02_manual_projection')

The first two components retain **42.2%** of standardized meta-feature
variance. The colored groups overlap substantially, so the projection does not support a
simple two-dimensional decision rule. It is still useful diagnostically: local clusters show
datasets with similar size, imbalance, distributional shape, and structural complexity, while
isolated points identify tasks whose scale or geometry is unusual. PCA is unsupervised, so
separation in this figure is descriptive rather than a validation score.

## 6. Meta-models on hand-designed features

In [7]:
meta_results = evaluate_both_sources(meta_dataset)
meta_results.to_csv(RESULTS_DIR / 'meta_model_results.csv', index=False)
manual_results = meta_results[meta_results['Feature source'].eq('Manual')].copy()
show_table(
    manual_results[[
        'Meta-model', 'Splits', 'balanced_accuracy_mean', 'balanced_accuracy_sd',
        'macro_f1_mean', 'accuracy_mean'
    ]],
    'Repeated five-fold validation on hand-designed meta-features',
)

+---------------------------------------------------------------------------------------------------------------+
|                          Repeated five-fold validation on hand-designed meta-features                         |
+----------------------+--------+------------------------+----------------------+---------------+---------------+
| Meta-model           | Splits | balanced_accuracy_mean | balanced_accuracy_sd | macro_f1_mean | accuracy_mean |
+----------------------+--------+------------------------+----------------------+---------------+---------------+
| Most frequent        | 25     | 0.3333                 | 0.0000               | 0.1740        | 0.3533        |
| Logistic regression  | 25     | 0.4407                 | 0.0523               | 0.4348        | 0.4433        |
| 15-nearest neighbors | 25     | 0.5110                 | 0.0465               | 0.5084        | 0.5133        |
| Random forest        | 25     | 0.4982                 | 0.0615               | 0.4966

The naive most-frequent classifier reaches mean balanced accuracy
**0.333**. The strongest tested meta-model is
**15-nearest neighbors** at **0.511**
(split SD 0.047). Accuracy is also reported, but balanced
accuracy and macro F1 are more informative here because the algorithm-winner classes are not
equally frequent. Repeated folds reduce dependence on one lucky split; their SD is a dispersion
summary, not a confidence interval.

## 7. Meta-features of the manual meta-dataset

In [8]:
meta_characteristics = characterize_meta_datasets(meta_dataset)
meta_characteristics.to_csv(RESULTS_DIR / 'meta_dataset_characteristics.csv', index=False)
manual_object = meta_characteristics[
    meta_characteristics['Object'].eq('Manual meta-dataset / manual extractor')
]
manual_row = manual_object.iloc[0]
manual_characteristics_table = pd.DataFrame([
    {'Metric': 'Meta-examples', 'Value': manual_row['basic__n_instances']},
    {'Metric': 'Meta-features', 'Value': manual_row['basic__n_features']},
    {'Metric': 'Meta-classes', 'Value': manual_row['basic__n_classes']},
    {'Metric': 'Minority meta-class share', 'Value': manual_row['stat__minority_fraction']},
    {'Metric': 'Normalized meta-class entropy', 'Value': manual_row['stat__normalized_class_entropy']},
    {'Metric': 'First-PC variance share', 'Value': manual_row['struct__pca_first_component']},
    {'Metric': 'PC fraction for 95% variance', 'Value': manual_row['struct__pca_components_95_fraction']},
])
show_table(
    manual_characteristics_table,
    'The manual meta-dataset treated as a classification dataset',
)

+-------------------------------------------------------------+
| The manual meta-dataset treated as a classification dataset |
+---------------------------------------------+---------------+
| Metric                                      | Value         |
+---------------------------------------------+---------------+
| Meta-examples                               | 300.0000      |
| Meta-features                               | 19.0000       |
| Meta-classes                                | 3.0000        |
| Minority meta-class share                   | 0.3167        |
| Normalized meta-class entropy               | 0.9991        |
| First-PC variance share                     | 0.1905        |
| PC fraction for 95% variance                | 0.6316        |
+---------------------------------------------+---------------+


This second extraction changes the unit of analysis. The 300 OpenML tasks are now the
observations, the hand-designed dataset descriptors are predictors, and the winning base
algorithm is the target. The class-entropy and minority-share values therefore describe the
**algorithm preference distribution**, not any original OpenML class distribution.

## 8. PyMFE repetition

PyMFE supplies an independent implementation with general, statistical, and complexity
groups. General and statistical features are extracted on the full cleaned dataset.
Complexity features use a deterministic stratified sample capped at 500 rows because nearest-
neighbor complexity measures are quadratic; the actual sample size is logged but excluded
from the predictive feature matrix. Vector-valued measures are summarized by mean and SD.
A column observed on fewer than 20% of datasets is also excluded; missingness indicators for
the retained columns are fitted inside validation rather than computed from held-out data.

PyMFE standardizes numerical predictors before the statistical group is calculated. Therefore
the summaries of their means and standard deviations are fixed at approximately 0 and 1.
Their tiny residual variation is floating-point noise, so those four columns are excluded from
PCA and meta-models. This is a feature-level correction: no OpenML dataset is removed.

In [9]:
pymfe_catalog = pd.DataFrame([
    {'Group': group, 'Requested PyMFE features': ', '.join(features)}
    for group, features in PYMFE_FEATURES.items()
])
show_table(pymfe_catalog, 'Fixed PyMFE feature selection', max_width=70)
excluded_artifacts = pd.DataFrame({
    'Excluded column': sorted(PYMFE_NUMERICAL_ARTIFACTS),
    'Reason': ['Fixed by z-standardization; residual variation is rounding noise']
              * len(PYMFE_NUMERICAL_ARTIFACTS),
})
show_table(excluded_artifacts, 'PyMFE columns excluded after diagnosis', max_width=48)

pymfe_projection, pymfe_variance, pymfe_loadings = pca_projection(meta_dataset, pymfe_columns)
pymfe_extremes = extreme_feature_rows(meta_dataset, pymfe_columns)
pymfe_projection.to_csv(RESULTS_DIR / 'pymfe_projection.csv', index=False)
pymfe_loadings.to_csv(RESULTS_DIR / 'pymfe_pca_loadings.csv', index=False)
pymfe_extremes.to_csv(RESULTS_DIR / 'pymfe_extreme_datasets.csv', index=False)

quartiles = pymfe_projection[['PC1', 'PC2']].quantile([0.25, 0.75])
iqr = quartiles.loc[0.75] - quartiles.loc[0.25]
lower = quartiles.loc[0.25] - 1.5 * iqr
upper = quartiles.loc[0.75] + 1.5 * iqr
central = (
    pymfe_projection['PC1'].between(lower['PC1'], upper['PC1'])
    & pymfe_projection['PC2'].between(lower['PC2'], upper['PC2'])
)
central_range = pymfe_projection.loc[central, ['PC1', 'PC2']].agg(['min', 'max'])
x_padding = 0.04 * (central_range.loc['max', 'PC1'] - central_range.loc['min', 'PC1'])
y_padding = 0.04 * (central_range.loc['max', 'PC2'] - central_range.loc['min', 'PC2'])
fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.10,
    subplot_titles=(
        f'Central view · {int(central.sum())}/{len(pymfe_projection)} tasks inside IQR fences',
        'Full range · all tasks retained',
    ),
)
for column in [1, 2]:
    for algorithm in ALGORITHMS:
        visible = central if column == 1 else pd.Series(True, index=pymfe_projection.index)
        subset = pymfe_projection[
            pymfe_projection['best_algorithm'].eq(algorithm) & visible
        ]
        fig.add_trace(go.Scatter(
            x=subset['PC1'], y=subset['PC2'], mode='markers', name=algorithm,
            legendgroup=algorithm, showlegend=column == 1,
            marker=dict(color=COLORS[algorithm], symbol=SYMBOLS[algorithm], size=9,
                        opacity=1.0, line=dict(color='white', width=0.7)),
            customdata=subset[['dataset_name', 'dataset_id']],
            hovertemplate='%{customdata[0]} · OpenML %{customdata[1]}<br>'
                          'PC1=%{x:.2f}<br>PC2=%{y:.2f}<extra></extra>',
        ), row=1, col=column)
fig.update_xaxes(
    title_text=f'PC1 ({pymfe_variance[0]:.1%} variance)',
    range=[central_range.loc['min', 'PC1'] - x_padding,
           central_range.loc['max', 'PC1'] + x_padding], row=1, col=1,
)
fig.update_yaxes(
    title_text=f'PC2 ({pymfe_variance[1]:.1%} variance)',
    range=[central_range.loc['min', 'PC2'] - y_padding,
           central_range.loc['max', 'PC2'] + y_padding], row=1, col=1,
)
fig.update_xaxes(title_text=f'PC1 ({pymfe_variance[0]:.1%} variance)', row=1, col=2)
fig.update_yaxes(title_text=f'PC2 ({pymfe_variance[1]:.1%} variance)', row=1, col=2)
fig.update_layout(
    title='PyMFE meta-space: readable center and complete context<br>'
          '<sup>The left panel changes only the view limits; the right panel keeps every dataset</sup>',
    legend_title_text='Best base algorithm',
)
_ = show_and_save(fig, '03_pymfe_projection')

extreme_table = pymfe_extremes.rename(columns={
    'dataset_name': 'Dataset', 'best_algorithm': 'Winner',
    'robust_z': 'Robust z', 'driving_feature': 'Driving feature',
    'feature_value': 'Value',
})[['Dataset', 'Winner', 'Driving feature', 'Value', 'Robust z']]
show_table(extreme_table, 'Most extreme retained PyMFE profiles', precision=2, max_width=34)

+------------------------------------------------------------------------------------+
|                           Fixed PyMFE feature selection                            |
+-------------+----------------------------------------------------------------------+
| Group       | Requested PyMFE features                                             |
+-------------+----------------------------------------------------------------------+
| general     | attr_to_inst, freq_class, nr_attr, nr_cat, nr_class, nr_inst, nr_num |
| statistical | cor, kurtosis, mean, sd, skewness                                    |
| complexity  | f1, f2, n2, t2                                                       |
+-------------+----------------------------------------------------------------------+
+-----------------------------------------------------------------------------------+
|                       PyMFE columns excluded after diagnosis                      |
+--------------------------------+-----------

+-------------------------------------------------------------------------------------+
|                         Most extreme retained PyMFE profiles                        |
+--------------+---------------------+---------------------------+---------+----------+
| Dataset      | Winner              | Driving feature           | Value   | Robust z |
+--------------+---------------------+---------------------------+---------+----------+
| ada_prior    | Logistic regression | statistical__kurtosis__sd | 1021.70 | 236.84   |
| dis          | RBF SVM             | statistical__kurtosis__sd | 726.04  | 168.10   |
| allrep       | RBF SVM             | statistical__kurtosis__sd | 726.04  | 168.10   |
| sick         | Random forest       | statistical__kurtosis__sd | 725.33  | 167.93   |
| hypothyroid  | Random forest       | statistical__kurtosis__sd | 725.33  | 167.93   |
| Sick_numeric | Random forest       | statistical__kurtosis__sd | 720.35  | 166.78   |
| ada          | Logistic regres

The PyMFE projection retains **36.3%** of variance in its first two
components. The left panel shows points inside the usual 1.5-IQR fences on both axes, making
the dense center readable; the right panel proves that all 300 datasets remain in the analysis.
It is not a second fit or a deletion rule. The remaining extremes are genuine high-kurtosis profiles,
concentrated in related sparse medical datasets, rather than floating-point artifacts. Their
presence is reported instead of being silently winsorized.

As with the manual representation, the winner classes overlap. PyMFE's `f1`, `f2`, and `n2`
emphasize class-boundary complexity, whereas the manual representation also includes duplicate
structure, outlier prevalence, and a direct Fisher-ratio aggregate.

In [10]:
pymfe_results = meta_results[meta_results['Feature source'].eq('PyMFE')].copy()
show_table(
    pymfe_results[[
        'Meta-model', 'Splits', 'balanced_accuracy_mean', 'balanced_accuracy_sd',
        'macro_f1_mean', 'accuracy_mean'
    ]],
    'Repeated five-fold validation on PyMFE meta-features',
)

fig = go.Figure()
model_order = ['Most frequent', 'Logistic regression', '15-nearest neighbors', 'Random forest']
model_position = {model: 3 - index for index, model in enumerate(model_order)}
for source in ['Manual', 'PyMFE']:
    subset = meta_results[meta_results['Feature source'].eq(source)]
    offset = 0.08 if source == 'Manual' else -0.08
    fig.add_trace(go.Scatter(
        x=subset['balanced_accuracy_mean'],
        y=subset['Meta-model'].map(model_position) + offset,
        error_x=dict(type='data', array=subset['balanced_accuracy_sd'], visible=True),
        mode='markers', name=source,
        marker=dict(color=SOURCE_COLORS[source], size=15,
                    symbol='circle' if source == 'Manual' else 'diamond',
                    line=dict(color='white', width=1)),
        customdata=subset[['macro_f1_mean', 'accuracy_mean']],
        text=subset['Meta-model'],
        hovertemplate='%{text}<br>Balanced accuracy=%{x:.3f}<br>'
                      'Macro F1=%{customdata[0]:.3f}<br>'
                      'Accuracy=%{customdata[1]:.3f}<extra></extra>',
    ))
fig.update_xaxes(title='Mean meta-level balanced accuracy', range=[0, 1], tickformat='.0%')
fig.update_yaxes(
    title='Meta-model', tickmode='array',
    tickvals=[model_position[model] for model in model_order],
    ticktext=model_order,
    range=[-0.45, 3.45],
)
fig.update_layout(
    title='Learned meta-models improve on the naive rule, with uncertainty across splits<br>'
          '<sup>5 × repeated stratified five-fold CV · error bars show split SD, not a confidence interval</sup>',
    legend_title_text='Meta-feature source',
)
_ = show_and_save(fig, '04_meta_model_comparison')

+---------------------------------------------------------------------------------------------------------------+
|                              Repeated five-fold validation on PyMFE meta-features                             |
+----------------------+--------+------------------------+----------------------+---------------+---------------+
| Meta-model           | Splits | balanced_accuracy_mean | balanced_accuracy_sd | macro_f1_mean | accuracy_mean |
+----------------------+--------+------------------------+----------------------+---------------+---------------+
| Most frequent        | 25     | 0.3333                 | 0.0000               | 0.1740        | 0.3533        |
| Logistic regression  | 25     | 0.4216                 | 0.0460               | 0.4182        | 0.4247        |
| 15-nearest neighbors | 25     | 0.5338                 | 0.0345               | 0.5298        | 0.5360        |
| Random forest        | 25     | 0.5317                 | 0.0566               | 0.5308

With PyMFE, the strongest tested meta-model is **15-nearest neighbors** with mean
balanced accuracy **0.534**, compared with
**0.333** for the naive rule. The manual and PyMFE
results answer the same prediction question under identical folds; any difference therefore
reflects the dataset representation and its interaction with the meta-model, not a different
validation split.

## 9. Meta-features of the PyMFE meta-dataset

In [11]:
pymfe_object = meta_characteristics[
    meta_characteristics['Object'].eq('PyMFE meta-dataset / PyMFE extractor')
]
pymfe_row = pymfe_object.iloc[0]
pymfe_characteristics_table = pd.DataFrame([
    {'Metric': 'Meta-examples', 'Value': pymfe_row['pymfe__general__nr_inst']},
    {'Metric': 'PyMFE meta-features', 'Value': pymfe_row['pymfe__general__nr_attr']},
    {'Metric': 'Meta-classes', 'Value': pymfe_row['pymfe__general__nr_class']},
    {'Metric': 'Mean meta-class frequency', 'Value': pymfe_row['pymfe__general__freq_class__mean']},
    {'Metric': 'SD of meta-class frequencies', 'Value': pymfe_row['pymfe__general__freq_class__sd']},
    {'Metric': 'Mean F1 complexity', 'Value': pymfe_row['pymfe__complexity__f1__mean']},
    {'Metric': 'Mean N2 complexity', 'Value': pymfe_row['pymfe__complexity__n2__mean']},
])
show_table(
    pymfe_characteristics_table,
    'The PyMFE meta-dataset characterized again by PyMFE',
)

+-----------------------------------------------------+
| The PyMFE meta-dataset characterized again by PyMFE |
+---------------------------------------+-------------+
| Metric                                | Value       |
+---------------------------------------+-------------+
| Meta-examples                         | 300.0000    |
| PyMFE meta-features                   | 20.0000     |
| Meta-classes                          | 3.0000      |
| Mean meta-class frequency             | 0.3333      |
| SD of meta-class frequencies          | 0.0186      |
| Mean F1 complexity                    | 0.9826      |
| Mean N2 complexity                    | 0.4615      |
+---------------------------------------+-------------+


PyMFE now sees a small, entirely numerical classification problem. Its general counts refer
to meta-examples and PyMFE columns, while complexity measures describe how algorithm-winner
classes overlap in this meta-feature space. They should not be compared numerically with the
same measures on an original dataset without keeping that change of level explicit.

## 10. Conclusions

- All manual and PyMFE features passed the row-, column-, and category-order tests; the largest
  numerical discrepancy was 3.55e-15.
- The base portfolio is not redundant. Random forest wins 106/300
  tasks, while the remaining tasks supply examples for the other meta-classes. At the same time,
  34.7% of labels come from gaps no larger than 0.01, so winner prediction has a
  genuine noise floor.
- Two-dimensional PCA preserves 42.2% of variance for the manual features
  and 36.3% for PyMFE. Four numerically meaningless PyMFE summaries were
  removed after diagnosis; no dataset was removed. Both pictures still show overlap, which is
  consistent with the imperfect meta-level validation scores.
- The best manual-feature result is 15-nearest neighbors
  (0.511 balanced accuracy); the best PyMFE result is
  15-nearest neighbors (0.534). Both are evaluated
  against the same naive majority-class reference.

These findings are conditional on the OpenML cohort, fixed preprocessing, fixed hyperparameters,
and balanced accuracy. Several task names belong to related synthetic or financial-series families,
so ordinary dataset-level CV may be optimistic for a completely new dataset family. The analysis
supports portfolio selection under this protocol; it does not establish that one meta-feature set
or base algorithm is universally superior.

## References

- [OpenML task API](https://docs.openml.org/reference/tasks/)
- [PyMFE documentation](https://pymfe.readthedocs.io/en/latest/generated/pymfe.mfe.MFE.html)
- [Balanced accuracy in scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html)
- [MLflow Tracking API](https://mlflow.org/docs/latest/tracking/tracking-api/)